In [ ]:
!pip install evaluate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np

In [ ]:
# load dataset
raw_datasets = load_dataset("flowaicom/HaluEval")

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets = raw_datasets.map(label_map)
print(f"Dataset loaded: {len(raw_datasets['test'])} rows")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset loaded: 10000 rows


In [ ]:
split_datasets = raw_datasets["test"].train_test_split(test_size=0.2, seed=42)

small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset  = split_datasets["test"].shuffle(seed=42).select(range(100))

print(f"Train size: {len(small_train_dataset)}")
print(f"Eval size:  {len(small_eval_dataset)}")

Train size: 400
Eval size:  100


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
MAX_LENGTH = 256

def tokenize_function(examples):
    return tokenizer(
        ["Question: " + q for q in examples["question"]],
        ["Answer: "   + a for a in examples["answer"]],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = small_train_dataset.map(tokenize_function, batched=True, load_from_cache_file=False)
eval_tokenized  = small_eval_dataset.map(tokenize_function, batched=True, load_from_cache_file=False)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs=8, learning_rate=2e-5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.134310,0.970000,0.942308,1.000000,0.970297
2,No log,0.153729,0.970000,0.942308,1.000000,0.970297
3,No log,0.179581,0.970000,0.942308,1.000000,0.970297
4,No log,0.179473,0.970000,0.942308,1.000000,0.970297
5,No log,0.218402,0.970000,0.942308,1.000000,0.970297
6,No log,0.232988,0.970000,0.942308,1.000000,0.970297
7,No log,0.231718,0.970000,0.942308,1.000000,0.970297
8,No log,0.227460,0.970000,0.942308,1.000000,0.970297


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.2274598330259323,
 'eval_accuracy': 0.97,
 'eval_precision': 0.9423076923076923,
 'eval_recall': 1.0,
 'eval_f1': 0.9702970297029703,
 'eval_runtime': 1.6445,
 'eval_samples_per_second': 60.81,
 'eval_steps_per_second': 7.905,
 'epoch': 8.0}